Supplementary Code S9
Locked-test validation under train-only model selection and calibration
Purpose

This notebook extends the revised, leakage-aware modeling framework established in Supplementary Code S8 by introducing a single, fully held-out locked test set that is excluded from all stages of model development.
The analysis is designed to approximate deployment-like evaluation conditions and to verify that model performance observed under nested cross-validation is not inflated by repeated data reuse.

The primary goal of S9 is to provide a one-time, unbiased assessment of generalization performance under strict train-only decision making.

Relationship to prior scripts

S2 defines the original nested cross-validation protocol and identifies preliminary winner models.

S5–S6 summarize performance–complexity trade-offs and audit potential leakage and dominant predictors.

S7 evaluates robustness to feature removal under fixed S2 configurations.

S8 corrects feature preprocessing and establishes a leakage-aware internal validation baseline using nested cross-validation.

S9 builds directly on S8 by adding an independent locked test set and enforcing strict separation between model development and final evaluation.

Key principles

Uses exactly the same target definition, cohort filtering, and governed feature sets as in S8.

Introduces a single locked test split that is never accessed during model selection, hyperparameter tuning, feature selection, threshold optimization, or calibration.

All modeling decisions are made exclusively on the training subset.

Final evaluation on the locked test set is performed once, without iteration or re-tuning.

Performance estimates from the locked test are interpreted as deployment-like but sample-size–limited diagnostics.

Methods implemented in S9

For each feature-set variant (FULL, CLINICAL, BIOMARKERS), the notebook performs:

Locked test split

The dataset is stratified into:

a training set, used for all model development steps, and

a locked test set, withheld until final evaluation.

The indices defining this split are saved and reused consistently in subsequent analyses.

Train-only model selection

Within the training set:

model classes and hyperparameters are selected using nested cross-validation,

optional feature reduction (e.g., RFE) is applied only within training folds,

out-of-fold (OOF) predictions are generated for unbiased internal performance estimation.

No information from the locked test set is used at any point in this process.

Threshold selection and calibration

Decision thresholds are selected exclusively on the training set using OOF predictions.
Probability calibration (e.g., isotonic regression) is applied using cross-fitted procedures restricted to training data, ensuring that calibrated probabilities remain leakage-free.

Final model refitting and locked-test evaluation

Final models are refit on the full training set using decisions fixed during train-only selection.
Discrimination and calibration metrics are then evaluated once on the locked test set.

Inputs (from S2–S8 workflow)

Required:

processed_dataset.csv

features_used_FULL.csv

features_used_CLINICAL.csv

features_used_BIOMARKERS.csv

Optional:

audit_nonnumeric_features.csv

All inputs are identical to, or derived from, those used in S8, ensuring continuity of feature governance and preprocessing.

Outputs

S9_locked_split_indices.csv — indices defining the training and locked test subsets.

S9_variant_matrix_diagnostics.csv — diagnostic summary of effective feature counts after preprocessing.

S9_trainonly_selection_details.csv — detailed record of train-only model selection, hyperparameters, thresholds, and calibration settings.

S9_locked_test_results.csv — discrimination and calibration metrics evaluated once on the locked test set.

S9_run_log.json — fully auditable log of inputs, configuration, and modeling decisions.

Interpretation note

Supplementary Code S9 provides a stringent test of generalization under realistic deployment constraints.
Near-perfect discrimination on the locked test set should be interpreted cautiously, particularly given the limited sample size, and treated as a diagnostic signal warranting further leakage auditing rather than evidence of clinical readiness.
Statistical robustness of these findings is evaluated separately in Supplementary Code S10.

## Setup + catalogs


In [1]:
from __future__ import annotations

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold, train_test_split, GridSearchCV, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, matthews_corrcoef, brier_score_loss

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Skipping features without any observed values.*")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

OUT_DIR = ROOT / "results" / "S9_locked_test"
TAB_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
for d in (TAB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Writing outputs to:", OUT_DIR.resolve())


ROOT: /content
Writing outputs to: /content/results/S9_locked_test


## Finder + loadery

In [2]:
SEARCH_ROOTS = [
    ROOT,
    ROOT / "results",
    ROOT / "results" / "S2_nestedcv",
    ROOT / "results" / "S6_leakage_audit",
    ROOT / "results" / "S8_revised_models",
    Path("/mnt/data"),
    ROOT / "drive",
    ROOT / "drive" / "MyDrive",
]

def find_file(filename: str, roots: list[Path]) -> Path:
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.rglob(filename))
            if hits:
                hits = sorted(hits, key=lambda x: len(str(x)))
                return hits[0]
    raise FileNotFoundError(
        f"Missing required file: {filename}\nSearched in:\n - " + "\n - ".join(map(str, roots))
    )

def read_csv_required(name: str) -> tuple[pd.DataFrame, Path]:
    p = find_file(name, SEARCH_ROOTS)
    df = pd.read_csv(p)
    print(f"Loaded: {p} | shape={df.shape}")
    return df, p

def load_feature_list(path: Path) -> list[str]:
    fdf = pd.read_csv(path)
    fdf.columns = [c.strip() for c in fdf.columns]
    for col in ["feature", "features", "feature_name", "column", "variable", "var", "name"]:
        if col in fdf.columns:
            feats = fdf[col].astype(str).str.strip().tolist()
            return [f for f in feats if f and f.lower() != "nan"]
    feats = fdf.iloc[:, 0].astype(str).str.strip().tolist()
    return [f for f in feats if f and f.lower() != "nan"]


## Configuration

In [3]:
TEST_SIZE = 0.20

N_OUTER = 5
N_INNER = 5

USE_RFE = True
RFE_N_SELECT = 10

USE_CALIBRATION = True
CALIB_METHOD = "isotonic"
CALIB_CV = 5

cv_outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
cv_inner = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)

print(
    "S9 config:",
    f"TEST_SIZE={TEST_SIZE}, OUTER={N_OUTER}, INNER={N_INNER}, USE_RFE={USE_RFE}, RFE_N_SELECT={RFE_N_SELECT},",
    f"CALIB={USE_CALIBRATION}({CALIB_METHOD}), CALIB_CV={CALIB_CV}"
)


S9 config: TEST_SIZE=0.2, OUTER=5, INNER=5, USE_RFE=True, RFE_N_SELECT=10, CALIB=True(isotonic), CALIB_CV=5


## Data + governed feature lists

In [4]:
df, path_processed = read_csv_required("processed_dataset.csv")

p_full = find_file("features_used_FULL.csv", SEARCH_ROOTS)
p_clin = find_file("features_used_CLINICAL.csv", SEARCH_ROOTS)
p_bio  = find_file("features_used_BIOMARKERS.csv", SEARCH_ROOTS)

features_used = {
    "FULL": load_feature_list(p_full),
    "CLINICAL": load_feature_list(p_clin),
    "BIOMARKERS": load_feature_list(p_bio),
}

print("Governed feature counts:", {k: len(v) for k, v in features_used.items()})


Loaded: /content/processed_dataset.csv | shape=(152, 117)
Governed feature counts: {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 10}


## Target

In [5]:
TARGET_COL = "typ_zawalu"
if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in processed_dataset.csv")

labels = df[TARGET_COL].astype(str).str.strip()
allowed = {"STEMI", "NSTEMI"}

df = df.loc[labels.isin(allowed)].copy()
y = (df[TARGET_COL].astype(str).str.strip() == "STEMI").astype(int).values

print("TARGET_COL:", TARGET_COL)
print("Task: STEMI vs NSTEMI (positive=STEMI)")
print(pd.Series(y).value_counts())
print("Positive rate:", float(np.mean(y)))

for k in features_used:
    features_used[k] = [f for f in features_used[k] if f != TARGET_COL]


TARGET_COL: typ_zawalu
Task: STEMI vs NSTEMI (positive=STEMI)
0    32
1    25
Name: count, dtype: int64
Positive rate: 0.43859649122807015


## Nonnumeric audit

In [6]:
NONNUMERIC_FEATURES = set()
try:
    audit_df, path_audit = read_csv_required("audit_nonnumeric_features.csv")
    cand = ["feature","feature_name","column","variable","var","name"]
    col = next((c for c in cand if c in audit_df.columns), None)
    if col is None:
        col = audit_df.columns[0]
    NONNUMERIC_FEATURES = set(audit_df[col].astype(str).tolist())
    print("Loaded NONNUMERIC_FEATURES:", len(NONNUMERIC_FEATURES))
except FileNotFoundError:
    path_audit = None
    print("NOTE: audit_nonnumeric_features.csv not found (OK).")

YES_SET = {"yes","y","true","t","1","tak","on","present","positive","pos"}
NO_SET  = {"no","n","false","f","0","nie","off","absent","negative","neg"}

def _normalize_str_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.strip()
             .str.lower()
             .str.replace(",", ".", regex=False))

def _try_map_binary_object(col: pd.Series) -> tuple[pd.Series, bool]:
    norm = _normalize_str_series(col)
    uniq = sorted(norm.dropna().unique().tolist())
    if len(uniq) == 0:
        return col, False
    if len(set(uniq)) <= 2 and all(u in (YES_SET | NO_SET) for u in uniq):
        def map_one(v):
            if pd.isna(v): return np.nan
            vv = str(v).strip().lower()
            if vv in YES_SET: return 1
            if vv in NO_SET:  return 0
            return np.nan
        return norm.map(map_one).astype(float), True
    return col, False


NOTE: audit_nonnumeric_features.csv not found (OK).


## Construction of the X matrix for variants + diagnostics

In [7]:
def build_matrix(df_in: pd.DataFrame, feats: list[str], variant_name: str):
    feats = [f for f in feats if f in df_in.columns and f != TARGET_COL]
    feats = [f for f in feats if f not in NONNUMERIC_FEATURES]

    Xdf = df_in[feats].copy()

    recoded = 0
    for c in Xdf.columns:
        if Xdf[c].dtype == "object" or str(Xdf[c].dtype).startswith("bool"):
            mapped, ok = _try_map_binary_object(Xdf[c])
            if ok:
                Xdf[c] = mapped
                recoded += 1

    for c in Xdf.columns:
        if Xdf[c].dtype == "object":
            Xdf[c] = _normalize_str_series(Xdf[c])
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")

    all_nan = Xdf.columns[Xdf.isna().mean() == 1.0].tolist()
    constant = Xdf.columns[Xdf.nunique(dropna=True) <= 1].tolist()
    dropped = sorted(set(all_nan + constant))

    if dropped:
        pd.DataFrame({"feature": dropped}).to_csv(
            TAB_DIR / f"S9_{variant_name}_dropped_empty_or_constant.csv", index=False
        )
        Xdf = Xdf.drop(columns=dropped)

    diag = {
        "variant": variant_name,
        "n_governed_present": int(len(feats)),
        "n_recoded_binary": int(recoded),
        "n_dropped_empty_or_constant": int(len(dropped)),
        "n_used_matrix": int(Xdf.shape[1]),
    }
    return Xdf.values, np.array(Xdf.columns.tolist()), diag

variants_cfg = {}
diag_rows = []
for v in ["FULL","CLINICAL","BIOMARKERS"]:
    Xv, fn, diag = build_matrix(df, features_used[v], v)
    variants_cfg[v] = {"X": Xv, "features": fn}
    diag_rows.append(diag)

diag_df = pd.DataFrame(diag_rows).sort_values("variant").reset_index(drop=True)
diag_df.to_csv(TAB_DIR / "S9_variant_matrix_diagnostics.csv", index=False)
print("Saved:", TAB_DIR / "S9_variant_matrix_diagnostics.csv")

try:
    display(diag_df)
except NameError:
    print(diag_df)

bad = diag_df[diag_df["n_used_matrix"] < 2]
if len(bad) > 0:
    raise RuntimeError("S9 cannot run: some variants have <2 usable predictors.\n" + bad.to_string(index=False))


Saved: /content/results/S9_locked_test/tables/S9_variant_matrix_diagnostics.csv


,variant,n_governed_present,n_recoded_binary,n_dropped_empty_or_constant,n_used_matrix
0,BIOMARKERS,10,0,0,10
1,CLINICAL,58,20,16,42
2,FULL,69,20,16,53


## Locked split (TRAIN vs LOCKED_TEST) + index write

In [8]:
idx = np.arange(len(y))
idx_tr, idx_te = train_test_split(
    idx, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

split_df = pd.DataFrame({
    "idx": idx,
    "split": np.where(np.isin(idx, idx_te), "LOCKED_TEST", "TRAIN"),
})
split_path = TAB_DIR / "S9_locked_split_indices.csv"
split_df.to_csv(split_path, index=False)

print("Locked split created:")
print("  TRAIN:", len(idx_tr))
print("  LOCKED_TEST:", len(idx_te))
print("Saved:", split_path)


Locked split created:
  TRAIN: 45
  LOCKED_TEST: 12
Saved: /content/results/S9_locked_test/tables/S9_locked_split_indices.csv


## Models + grids

In [9]:
REVISED_MODELS = {
    "LR_L1": LogisticRegression(penalty="l1", solver="saga", max_iter=8000, random_state=RANDOM_STATE),
    "LR_ELNET": LogisticRegression(penalty="elasticnet", solver="saga", max_iter=8000, random_state=RANDOM_STATE),
    "SVM_REG": SVC(probability=True, random_state=RANDOM_STATE),
    "RF_SHALLOW": RandomForestClassifier(random_state=RANDOM_STATE),
}

REVISED_GRIDS = {
    "LR_L1": {"clf__C": np.logspace(-3, 2, 10)},
    "LR_ELNET": {"clf__C": np.logspace(-3, 2, 10), "clf__l1_ratio": [0.1, 0.5, 0.9]},
    "SVM_REG": {"clf__C": np.logspace(-3, 2, 10), "clf__kernel": ["linear","rbf"], "clf__gamma": ["scale"]},
    "RF_SHALLOW": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [2,3,4,5],
        "clf__min_samples_leaf": [5,10,20],
        "clf__max_features": ["sqrt", 0.5],
    },
}


## Pipeline builder

In [10]:
def make_pipeline(model_key: str, use_rfe: bool, n_features_to_select: int):
    pre_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_key in ["LR_L1","LR_ELNET","SVM_REG"]:
        pre_steps.append(("scaler", StandardScaler()))
    preprocess = Pipeline(pre_steps)

    steps = [("preprocess", preprocess)]

    if use_rfe:
        rfe_est = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
        steps.append(("rfe", RFE(estimator=rfe_est, n_features_to_select=n_features_to_select)))

    steps.append(("clf", REVISED_MODELS[model_key]))
    return Pipeline(steps)


## Threshold MCC helper

In [11]:
def choose_threshold_by_mcc(y_true, probs):
    ths = np.linspace(0.01, 0.99, 99)
    best_mcc, best_thr = -999.0, 0.5
    for t in ths:
        pred = (probs >= t).astype(int)
        mcc = matthews_corrcoef(y_true, pred)
        if mcc > best_mcc:
            best_mcc, best_thr = float(mcc), float(t)
    return best_thr, best_mcc


## Nested OOF on TRAIN

In [12]:
def nested_oof_on_train(X_tr, y_tr, model_key: str, use_rfe: bool):
    oof = np.zeros(len(y_tr), dtype=float)
    aucs, mccs = [], []
    fold_best_params = []

    n_select = min(RFE_N_SELECT, X_tr.shape[1]-1) if use_rfe else None
    if use_rfe and (n_select is None or n_select < 2):
        use_rfe = False
        n_select = None

    for fold_id, (itr, iva) in enumerate(cv_outer.split(X_tr, y_tr), start=1):
        pipe = make_pipeline(model_key, use_rfe=use_rfe, n_features_to_select=(n_select or RFE_N_SELECT))
        gs = GridSearchCV(pipe, REVISED_GRIDS[model_key], scoring="roc_auc", cv=cv_inner, n_jobs=-1, refit=True)
        gs.fit(X_tr[itr], y_tr[itr])

        best_model = gs.best_estimator_
        fold_best_params.append(gs.best_params_)

        probs = best_model.predict_proba(X_tr[iva])[:, 1]
        oof[iva] = probs

        aucs.append(roc_auc_score(y_tr[iva], probs))
        mccs.append(matthews_corrcoef(y_tr[iva], (probs >= 0.5).astype(int)))


    thr, thr_mcc = choose_threshold_by_mcc(y_tr, oof)

    return {
        "oof_probs": oof,
        "auc_mean_outer": float(np.mean(aucs)),
        "mcc_mean_outer@0.5": float(np.mean(mccs)),
        "thr_mcc_train_raw_oof": float(thr),
        "mcc_train@thr_raw_oof": float(thr_mcc),
        "fold_best_params": fold_best_params,
        "use_rfe_effective": bool(use_rfe),
        "n_select_effective": int(n_select) if use_rfe else None
    }


## Refit best on full TRAIN (inner CV)

In [13]:
def refit_best_on_full_train(X_tr, y_tr, model_key: str, use_rfe: bool):
    n_select = min(RFE_N_SELECT, X_tr.shape[1]-1) if use_rfe else None
    if use_rfe and (n_select is None or n_select < 2):
        use_rfe = False
        n_select = None

    pipe = make_pipeline(model_key, use_rfe=use_rfe, n_features_to_select=(n_select or RFE_N_SELECT))
    gs = GridSearchCV(pipe, REVISED_GRIDS[model_key], scoring="roc_auc", cv=cv_inner, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)

    return gs.best_estimator_, gs.best_params_, float(gs.best_score_), bool(use_rfe), (int(n_select) if use_rfe else None)


## Helpers base pipeline

In [14]:
def build_unfitted_best_pipeline(model_key: str, use_rfe: bool, X_tr: np.ndarray, best_params: dict):
    n_select = min(RFE_N_SELECT, X_tr.shape[1]-1) if use_rfe else None
    if use_rfe and (n_select is None or n_select < 2):
        use_rfe = False
        n_select = None

    pipe = make_pipeline(model_key, use_rfe=use_rfe, n_features_to_select=(n_select or RFE_N_SELECT))
    pipe.set_params(**best_params)
    return pipe, bool(use_rfe), (int(n_select) if use_rfe else None)


def calibrated_oof_probs(base_pipeline, X_tr, y_tr, *, calib_method: str, calib_cv: int, oof_cv):
    cal = CalibratedClassifierCV(
        estimator=clone(base_pipeline),
        method=calib_method,
        cv=calib_cv
    )
    oof_proba = cross_val_predict(
        cal,
        X_tr,
        y_tr,
        cv=oof_cv,
        method="predict_proba"
    )[:, 1]
    return oof_proba


def select_train_threshold_consistent(
    X_tr, y_tr,
    *,
    model_key: str,
    use_rfe: bool,
    best_params: dict,
    use_calibration: bool,
    calib_method: str,
    calib_cv: int,
    oof_cv
):
    base_pipe, use_rfe_eff, n_select_eff = build_unfitted_best_pipeline(
        model_key=model_key,
        use_rfe=use_rfe,
        X_tr=X_tr,
        best_params=best_params
    )

    if use_calibration:
        oof_probs = calibrated_oof_probs(
            base_pipe, X_tr, y_tr,
            calib_method=calib_method,
            calib_cv=calib_cv,
            oof_cv=oof_cv
        )
        thr_source = "calibrated_oof"
    else:
        oof_probs = cross_val_predict(
            clone(base_pipe),
            X_tr,
            y_tr,
            cv=oof_cv,
            method="predict_proba"
        )[:, 1]
        thr_source = "raw_oof"

    thr, thr_mcc = choose_threshold_by_mcc(y_tr, oof_probs)

    return {
        "thr": float(thr),
        "thr_mcc": float(thr_mcc),
        "thr_source": thr_source,
        "use_rfe_effective": use_rfe_eff,
        "n_select_effective": n_select_eff,
    }


## Main loop: selection on TRAIN + fit final + LOCKED_TEST + save CSV

In [15]:
rows_results = []
rows_details = []

for variant_name in ["FULL","CLINICAL","BIOMARKERS"]:
    Xv = variants_cfg[variant_name]["X"]
    feats_v = variants_cfg[variant_name]["features"]

    X_tr, y_tr = Xv[idx_tr], y[idx_tr]
    X_te, y_te = Xv[idx_te], y[idx_te]

    print("\n==============================")
    print("S9 Train-only selection:", variant_name)
    print("==============================")
    print("X_train:", X_tr.shape, "X_test_locked:", X_te.shape)


    candidates = []
    for model_key in REVISED_MODELS.keys():
        info_plain = nested_oof_on_train(X_tr, y_tr, model_key=model_key, use_rfe=False)
        candidates.append((model_key, False, info_plain["auc_mean_outer"], info_plain))

        if USE_RFE:
            info_rfe = nested_oof_on_train(X_tr, y_tr, model_key=model_key, use_rfe=True)
            candidates.append((model_key, True, info_rfe["auc_mean_outer"], info_rfe))

    candidates_sorted = sorted(candidates, key=lambda x: x[2], reverse=True)
    best_model_key, best_use_rfe, best_auc_train, best_info = candidates_sorted[0]

    print(
        "Best (train-only):",
        best_model_key,
        "RFE" if best_info["use_rfe_effective"] else "NO_RFE",
        "AUC_outer_mean=", round(best_auc_train, 4)
    )


    best_estimator, best_params_fulltrain, best_inner_auc, use_rfe_eff, n_select_eff = refit_best_on_full_train(
        X_tr, y_tr, model_key=best_model_key, use_rfe=best_use_rfe
    )


    oof_cv_thr = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)

    thr_pack = select_train_threshold_consistent(
        X_tr, y_tr,
        model_key=best_model_key,
        use_rfe=bool(use_rfe_eff),
        best_params=best_params_fulltrain,
        use_calibration=USE_CALIBRATION,
        calib_method=CALIB_METHOD,
        calib_cv=CALIB_CV,
        oof_cv=oof_cv_thr
    )

    thr = thr_pack["thr"]
    thr_mcc_train = thr_pack["thr_mcc"]
    thr_source = thr_pack["thr_source"]


    base_pipe, _, _ = build_unfitted_best_pipeline(
        model_key=best_model_key,
        use_rfe=bool(use_rfe_eff),
        X_tr=X_tr,
        best_params=best_params_fulltrain
    )

    if USE_CALIBRATION:
        final_model = CalibratedClassifierCV(
            estimator=clone(base_pipe),
            method=CALIB_METHOD,
            cv=CALIB_CV
        )
        final_model.fit(X_tr, y_tr)
    else:
        final_model = clone(base_pipe)
        final_model.fit(X_tr, y_tr)


    probs_test = final_model.predict_proba(X_te)[:, 1]
    auc_test = roc_auc_score(y_te, probs_test)
    brier_test = brier_score_loss(y_te, probs_test)

    pred_test = (probs_test >= thr).astype(int)
    mcc_test = matthews_corrcoef(y_te, pred_test)


    rows_details.append({
        "variant": variant_name,
        "selected_model": best_model_key,
        "selected_rfe_requested": bool(best_use_rfe),
        "selected_rfe_effective": bool(use_rfe_eff),
        "n_select_effective": n_select_eff,
        "train_outer_auc_mean_selected": float(best_auc_train),

        "train_thr_mcc_selected_on_oof": float(thr),
        "train_mcc_at_thr_on_oof": float(thr_mcc_train),
        "thr_source": thr_source,

        "refit_bestparams_fulltrain": json.dumps(best_params_fulltrain, ensure_ascii=False),
        "refit_inner_auc_mean": float(best_inner_auc),
        "calibration": (CALIB_METHOD if USE_CALIBRATION else "none"),
    })

    rows_results.append({
        "variant": variant_name,
        "selected_model": best_model_key + ("_RFE" if use_rfe_eff else ""),
        "locked_test_auc": float(auc_test),
        "locked_test_brier": float(brier_test),
        "locked_test_mcc_at_train_thr": float(mcc_test),

        "trainonly_outer_auc_mean": float(best_auc_train),
        "trainonly_thr_mcc": float(thr),
        "thr_source": thr_source,

        "n_train": int(len(idx_tr)),
        "n_test_locked": int(len(idx_te)),
        "n_features_effective_input": int(X_tr.shape[1]),
        "n_feature_names": int(len(feats_v)),
    })

df_results = pd.DataFrame(rows_results).sort_values(["variant"]).reset_index(drop=True)
df_details = pd.DataFrame(rows_details).sort_values(["variant"]).reset_index(drop=True)

res_path = TAB_DIR / "S9_locked_test_results.csv"
det_path = TAB_DIR / "S9_trainonly_selection_details.csv"

df_results.to_csv(res_path, index=False)
df_details.to_csv(det_path, index=False)

print("\nSaved outputs:")
print(" -", res_path)
print(" -", det_path)
print("\nLocked-test summary:")
try:
    display(df_results)
except NameError:
    print(df_results)



S9 Train-only selection: FULL
X_train: (45, 53) X_test_locked: (12, 53)
Best (train-only): LR_L1 NO_RFE AUC_outer_mean= 1.0

S9 Train-only selection: CLINICAL
X_train: (45, 42) X_test_locked: (12, 42)
Best (train-only): SVM_REG RFE AUC_outer_mean= 0.59

S9 Train-only selection: BIOMARKERS
X_train: (45, 10) X_test_locked: (12, 10)
Best (train-only): RF_SHALLOW NO_RFE AUC_outer_mean= 0.95

Saved outputs:
 - /content/results/S9_locked_test/tables/S9_locked_test_results.csv
 - /content/results/S9_locked_test/tables/S9_trainonly_selection_details.csv

Locked-test summary:


,variant,selected_model,locked_test_auc,locked_test_brier,locked_test_mcc_at_train_thr,trainonly_outer_auc_mean,trainonly_thr_mcc,thr_source,n_train,n_test_locked,n_features_effective_input,n_feature_names
0,BIOMARKERS,RF_SHALLOW,0.885714,0.105042,0.845154,0.95,0.33,calibrated_oof,45,12,10,10
1,CLINICAL,SVM_REG_RFE,0.628571,0.234443,0.000000,0.59,0.65,calibrated_oof,45,12,42,42
2,FULL,LR_L1,1.000000,0.000302,1.000000,1.00,0.38,calibrated_oof,45,12,53,53


## Logs

In [16]:
run_log = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "random_state": RANDOM_STATE,
    "target": {
        "TARGET_COL": TARGET_COL,
        "task": "STEMI vs NSTEMI (positive=STEMI)",
        "n_samples_after_filtering": int(len(y)),
        "positive_rate": float(np.mean(y)),
    },
    "split": {
        "TEST_SIZE": TEST_SIZE,
        "n_train": int(len(idx_tr)),
        "n_test_locked": int(len(idx_te)),
        "split_file": str(split_path),
    },
    "cv": {"outer": N_OUTER, "inner": N_INNER, "calibration_cv": CALIB_CV},
    "rfe": {"enabled": USE_RFE, "n_select_default": RFE_N_SELECT},
    "calibration": {"enabled": USE_CALIBRATION, "method": CALIB_METHOD},
    "inputs": {
        "processed_dataset.csv": str(path_processed),
        "features_used_FULL.csv": str(p_full),
        "features_used_CLINICAL.csv": str(p_clin),
        "features_used_BIOMARKERS.csv": str(p_bio),
        "audit_nonnumeric_features.csv": str(path_audit) if path_audit else None,
    },
    "outputs": {
        "S9_variant_matrix_diagnostics.csv": str(TAB_DIR / "S9_variant_matrix_diagnostics.csv"),
        "S9_locked_split_indices.csv": str(split_path),
        "S9_locked_test_results.csv": str(res_path),
        "S9_trainonly_selection_details.csv": str(det_path),
    },
    "notes": [
        "Locked test split is never used for tuning, feature selection, calibration, or threshold selection.",
        "Model selection and hyperparameter tuning are performed within TRAIN only using nested CV.",
        "Threshold is selected on TRAIN only using OOF predictions; if calibration is enabled, threshold selection uses calibrated OOF probabilities."
    ]
}

log_path = LOG_DIR / "S9_run_log.json"
log_path.write_text(json.dumps(run_log, indent=2), encoding="utf-8")
print("Saved:", log_path)


Saved: /content/results/S9_locked_test/logs/S9_run_log.json


/tmp/ipython-input-2400342605.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
